# Notebook 03 - Huấn luyện baseline và 4 model

Thực hiện riêng Bước 3. Đầu vào: dataset.zip. Đầu ra: training_artifacts.zip gồm 5 candidate pipeline, schema, kế hoạch tham số, môi trường và kết quả CV. Test được tách nhưng không đánh giá tại đây.

## 0. Chạy trên Colab

Chọn Runtime → Run all, upload dataset.zip và chờ GridSearchCV. Tải training_artifacts.zip ở cuối để đưa vào notebook 04.

In [1]:
from pathlib import Path
from google.colab import files
import io,json,shutil,sys,time,zipfile
import joblib
import numpy as np
import pandas as pd
import sklearn
ROOT=Path('/content/ckd_colab'); DATA_PATH=ROOT/'dataset.zip'
MODEL_DIR=ROOT/'models'; CANDIDATE_DIR=MODEL_DIR/'candidates'
for directory in [ROOT,MODEL_DIR,CANDIDATE_DIR]: directory.mkdir(parents=True,exist_ok=True)
if not DATA_PATH.is_file():
    print('Upload dataset.zip để chạy notebook 03.')
    uploaded=files.upload(); names=[n for n in uploaded if n.lower().endswith('.zip')]
    if len(names)!=1: raise FileNotFoundError('Cần đúng một dataset.zip.')
    DATA_PATH.write_bytes(uploaded[names[0]])
print('Python:',sys.version.split()[0])
print('pandas:',pd.__version__,'| numpy:',np.__version__)
print('scikit-learn:',sklearn.__version__,'| joblib:',joblib.__version__)

Python: 3.13.15
pandas: 2.2.3 | numpy: 2.1.3
scikit-learn: 1.6.1 | joblib: 1.6.0


## 1. Cấu hình phải giống notebook 02

Lặp lại hằng số giúp notebook chạy độc lập. Target, 24 feature, test_size và random_state không được thay đổi giữa các bước.

In [2]:
RANDOM_STATE=42; TEST_SIZE=0.20
TARGET='classification'; POSITIVE_LABEL='ckd'; NEGATIVE_LABEL='notckd'
NUMERIC_FEATURES=['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']
CATEGORICAL_FEATURES=['rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane']
FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES
assert len(FEATURES)==24 and 'id' not in FEATURES
print('Hợp đồng đặc trưng:',len(FEATURES),'features')

Hợp đồng đặc trưng: 24 features


## 2. Dùng lại đúng logic làm sạch

Không thay cách chuẩn hóa giữa bước 2 và 3. Mọi candidate sẽ là pipeline hoàn chỉnh preprocessor → estimator.

In [3]:
def read_zip_dataset(path):
    with zipfile.ZipFile(path) as archive:
        names=[n for n in archive.namelist() if n.lower().endswith('.csv')]
        if len(names)!=1: raise ValueError(f'ZIP phải có đúng 1 CSV: {names}')
        return pd.read_csv(io.BytesIO(archive.read(names[0])))

def clean_dataframe(dataframe):
    df = dataframe.copy()

    # Chuẩn hóa tên cột
    df.columns = [
        str(c).replace('\t', '').strip()
        for c in df.columns
    ]

    # Kiểm tra đủ cột
    missing = sorted(set(FEATURES + [TARGET]) - set(df.columns))
    if missing:
        raise ValueError(f'Thiếu cột: {missing}')

    # Chuẩn hóa các cột categorical/string
    for col in CATEGORICAL_FEATURES + [TARGET]:
        df[col] = df[col].map(
            lambda x: (
                np.nan
                if pd.isna(x) or str(x).strip() in ['', '?']
                else str(x).replace('\t', '').strip().lower()
            )
        ).astype(object)

    # Ép các cột số về numeric
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(
            df[col], errors='coerce'
        ).astype('float64')

    # Kiểm tra target
    invalid = set(df[TARGET].dropna().unique()) - {
        POSITIVE_LABEL,
        NEGATIVE_LABEL
    }

    if invalid:
        raise ValueError(f'Target không hợp lệ: {invalid}')

    # Xóa dòng trùng
    return df.drop_duplicates().reset_index(drop=True)

def build_preprocessor():
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder,StandardScaler
    numeric=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
    categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                          ('encoder',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric',numeric,NUMERIC_FEATURES),
                              ('categorical',categorical,CATEGORICAL_FEATURES)])

## 3. Tạo cùng stratified split

Mọi model dùng cùng X_train/y_train. X_test/y_test chỉ được tạo và giữ kín cho notebook 04; không xuất hiện trong lệnh predict/metric của bước train.

In [4]:
from sklearn.model_selection import train_test_split
df=clean_dataframe(read_zip_dataset(DATA_PATH))
X=df[FEATURES].copy(); y=df[TARGET].copy()
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y
)
print('Train dùng CV:',X_train.shape,'| Test giữ kín:',X_test.shape)
display(pd.DataFrame({'train':y_train.value_counts(),'test':y_test.value_counts()}))

Train dùng CV: (320, 24) | Test giữ kín: (80, 24)


,train,test
classification,,
ckd,200,50
notckd,120,30


## 4. Baseline và bốn model học máy

| Model | Nguyên lý | Tham số chính |
|---|---|---|
| Dummy | Luôn chọn lớp phổ biến, làm mốc tối thiểu | không tuning |
| Logistic Regression | biên tuyến tính/xác suất | C, class_weight |
| KNN | bỏ phiếu láng giềng | n_neighbors, weights, p |
| SVC | tối đa margin/kernel | C, kernel, gamma |
| Random Forest | bagging nhiều cây | số cây, độ sâu, min split |

Dummy chỉ là baseline, không tính thay cho bốn model chính.

## 5. Hàm tạo pipeline model

Tạo preprocessor mới cho mỗi estimator. GridSearchCV sẽ fit cả preprocessing trong từng fold train, tránh leakage sang fold validation.

In [5]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
def build_model_pipeline(estimator):
    return Pipeline([('preprocessor',build_preprocessor()),('model',clone(estimator))])

## 6. Không gian siêu tham số

Grid đủ đa dạng nhưng vẫn phù hợp dữ liệu 400 mẫu và thời gian Colab. Mọi lựa chọn tham số chỉ dựa trên CV train.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

model_specs={
 'dummy':(DummyClassifier(strategy='most_frequent',random_state=RANDOM_STATE),[{}]),
 'logistic_regression':(LogisticRegression(max_iter=1000,random_state=RANDOM_STATE),
   {'model__C':[0.01,0.1,1,10],'model__class_weight':[None,'balanced']}),
 'knn':(KNeighborsClassifier(),
   {'model__n_neighbors':[3,5,7,9],'model__weights':['uniform','distance'],'model__p':[1,2]}),
 'svc':(SVC(probability=True,random_state=RANDOM_STATE),
   {'model__C':[0.1,1,10],'model__kernel':['linear','rbf'],
    'model__gamma':['scale','auto'],'model__class_weight':[None,'balanced']}),
 'random_forest':(RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=-1),
   {'model__n_estimators':[100],'model__max_depth':[None,10],
    'model__min_samples_split':[2,5],'model__class_weight':[None,'balanced']})
}
parameter_plan={name:grid for name,(_,grid) in model_specs.items()}
print(json.dumps(parameter_plan,ensure_ascii=False,indent=2))

{
  "dummy": [
    {}
  ],
  "logistic_regression": {
    "model__C": [
      0.01,
      0.1,
      1,
      10
    ],
    "model__class_weight": [
      null,
      "balanced"
    ]
  },
  "knn": {
    "model__n_neighbors": [
      3,
      5,
      7,
      9
    ],
    "model__weights": [
      "uniform",
      "distance"
    ],
    "model__p": [
      1,
      2
    ]
  },
  "svc": {
    "model__C": [
      0.1,
      1,
      10
    ],
    "model__kernel": [
      "linear",
      "rbf"
    ],
    "model__gamma": [
      "scale",
      "auto"
    ],
    "model__class_weight": [
      null,
      "balanced"
    ]
  },
  "random_forest": {
    "model__n_estimators": [
      100
    ],
    "model__max_depth": [
      null,
      10
    ],
    "model__min_samples_split": [
      2,
      5
    ],
    "model__class_weight": [
      null,
      "balanced"
    ]
  }
}


## 7. GridSearchCV và metric

F1 của lớp ckd cân bằng Precision/Recall. CV 5-fold tận dụng train tốt hơn một validation split. Lưu mean/std CV, train score, thời gian fit/predict và kích thước file.

In [7]:
from sklearn.metrics import f1_score,make_scorer
from sklearn.model_selection import GridSearchCV
f1_scorer=make_scorer(f1_score,pos_label=POSITIVE_LABEL)

def train_candidate(name,estimator,param_grid):
    search=GridSearchCV(build_model_pipeline(estimator),param_grid,scoring=f1_scorer,
                        cv=5,n_jobs=-1,return_train_score=True,refit=True)
    started=time.perf_counter(); search.fit(X_train,y_train)
    train_seconds=time.perf_counter()-started
    fitted=search.best_estimator_; timing=[]
    for _ in range(7):
        start=time.perf_counter(); fitted.predict(X_train); timing.append(time.perf_counter()-start)
    path=CANDIDATE_DIR/f'{name}.joblib'; joblib.dump(fitted,path,compress=3)
    index=search.best_index_
    return {
      'model':name,'best_params':json.dumps(search.best_params_,sort_keys=True),
      'cv_f1':float(search.best_score_),
      'cv_f1_std':float(search.cv_results_['std_test_score'][index]),
      'cv_train_f1':float(search.cv_results_['mean_train_score'][index]),
      'train_f1':float(f1_score(y_train,fitted.predict(X_train),pos_label=POSITIVE_LABEL)),
      'train_seconds':train_seconds,
      'predict_ms_per_sample_train':float(np.median(timing)/len(X_train)*1000),
      'model_size_bytes':path.stat().st_size
    }

## 8. Huấn luyện từng candidate

Cell này có thể mất vài phút. Log cho biết model đang chạy. Test không được dùng.

In [8]:
rows=[]
for name,(estimator,grid) in model_specs.items():
    print('Đang huấn luyện:',name)
    row=train_candidate(name,estimator,grid); rows.append(row)
    print(f"Hoàn tất: CV F1={row['cv_f1']:.4f} ± {row['cv_f1_std']:.4f}")
assert len(rows)==5
print('Đủ Dummy + 4 model: OK')

Đang huấn luyện: dummy
Hoàn tất: CV F1=0.7692 ± 0.0000
Đang huấn luyện: logistic_regression
Hoàn tất: CV F1=1.0000 ± 0.0000
Đang huấn luyện: knn


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [       nan 0.97402163 0.96376522 0.96376522        nan 0.97429153
 0.96889342 0.96889342        nan 0.97429153 0.96889342 0.97162753
        nan 0.96889342 0.96636697 0.96636697]
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the train scores are non-finite: [       nan 1.         0.98410651 1.                nan 1.
 0.98281634 1.                nan 1.         0.97698042 1.
        nan 1.         0.97303135 1.        ]
  warnings.warn(


Hoàn tất: CV F1=0.9743 ± 0.0083
Đang huấn luyện: svc
Hoàn tất: CV F1=0.9975 ± 0.0051
Đang huấn luyện: random_forest
Hoàn tất: CV F1=0.9975 ± 0.0049
Đủ Dummy + 4 model: OK


## 9. Bảng kết quả CV

Đây chưa phải kết quả test. Dùng bảng để giải thích best params, ổn định giữa fold, gap train-CV, thời gian và kích thước.

In [9]:
results=pd.DataFrame(rows).sort_values('cv_f1',ascending=False).reset_index(drop=True)
results['cv_train_validation_gap']=results['cv_train_f1']-results['cv_f1']
display(results[['model','best_params','cv_f1','cv_f1_std','cv_train_f1',
                 'cv_train_validation_gap','train_seconds',
                 'predict_ms_per_sample_train','model_size_bytes']])
results.to_csv(MODEL_DIR/'training_results.csv',index=False)
results.to_json(MODEL_DIR/'training_results.json',orient='records',indent=2)
(MODEL_DIR/'parameter_plan.json').write_text(
    json.dumps(parameter_plan,ensure_ascii=False,indent=2),encoding='utf-8')

,model,best_params,cv_f1,cv_f1_std,cv_train_f1,cv_train_validation_gap,train_seconds,predict_ms_per_sample_train,model_size_bytes
0,logistic_regression,"{""model__C"": 1, ""model__class_weight"": null}",1.000000,0.000000,1.000000,0.000000,4.737241,0.137194,2665
1,random_forest,"{""model__class_weight"": ""balanced"", ""model__ma...",0.997531,0.004938,1.000000,0.002469,11.365859,0.140774,61050
2,svc,"{""model__C"": 0.1, ""model__class_weight"": null,...",0.997468,0.005063,0.999377,0.001909,9.944898,0.026142,5342
3,knn,"{""model__n_neighbors"": 5, ""model__p"": 1, ""mode...",0.974292,0.008318,1.000000,0.025708,8.211904,0.071163,17625
4,dummy,{},0.769231,0.000000,0.769231,0.000000,5.799942,0.052908,2206


878

## 10. Diễn giải CV

Model cần vượt Dummy; CV std nhỏ là ổn định; gap train-CV lớn cảnh báo overfitting. Chưa chọn model cuối cho tới khi xem test, FN/FP và chi phí deploy ở notebook 04.

In [10]:
baseline=float(results.loc[results.model=='dummy','cv_f1'].iloc[0])
for row in results.itertuples():
    status='tốt hơn Dummy' if row.cv_f1>baseline else 'baseline hoặc không hơn Dummy'
    print(f'{row.model}: CV F1={row.cv_f1:.3f} ± {row.cv_f1_std:.3f}; '
          f'gap={row.cv_train_validation_gap:.3f}; {status}.')
print('X_test chưa được dùng để tính metric: OK')

logistic_regression: CV F1=1.000 ± 0.000; gap=0.000; tốt hơn Dummy.
random_forest: CV F1=0.998 ± 0.005; gap=0.002; tốt hơn Dummy.
svc: CV F1=0.997 ± 0.005; gap=0.002; tốt hơn Dummy.
knn: CV F1=0.974 ± 0.008; gap=0.026; tốt hơn Dummy.
dummy: CV F1=0.769 ± 0.000; gap=0.000; baseline hoặc không hơn Dummy.
X_test chưa được dùng để tính metric: OK


## 11. Schema và phiên bản môi trường

Notebook 04 kiểm tra scikit-learn/joblib trước khi load. Nếu khác phiên bản, chạy lại notebook 03 và 04 cùng thời điểm.

In [11]:
schema={'task_type':'binary_classification','target':TARGET,'positive_label':POSITIVE_LABEL,
 'target_labels':[POSITIVE_LABEL,NEGATIVE_LABEL],'features':FEATURES,
 'numeric_features':NUMERIC_FEATURES,'categorical_features':CATEGORICAL_FEATURES,
 'allowed_categories':{col:sorted(df[col].dropna().astype(str).unique().tolist()) for col in CATEGORICAL_FEATURES},
 'numeric_ranges':{col:{'min':float(df[col].min()),'max':float(df[col].max())} for col in NUMERIC_FEATURES},
 'test_size':TEST_SIZE,'random_state':RANDOM_STATE}
(MODEL_DIR/'schema.json').write_text(json.dumps(schema,ensure_ascii=False,indent=2),encoding='utf-8')
environment={'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__,
             'scikit-learn':sklearn.__version__,'joblib':joblib.__version__}
(MODEL_DIR/'training_environment.json').write_text(json.dumps(environment,indent=2),encoding='utf-8')
print(json.dumps(environment,indent=2))

{
  "python": "3.13.15",
  "pandas": "2.2.3",
  "numpy": "2.1.3",
  "scikit-learn": "1.6.1",
  "joblib": "1.6.0"
}


## 12. Đóng gói cho notebook 04

training_artifacts.zip chứa 5 candidate và toàn bộ bảng/metadata; không cần tải riêng từng file.

In [12]:
archive_path=shutil.make_archive(str(ROOT/'training_artifacts'),'zip',MODEL_DIR)
with zipfile.ZipFile(archive_path) as archive: exported=set(archive.namelist())
required={'training_results.csv','training_results.json','parameter_plan.json',
          'schema.json','training_environment.json'}
assert required.issubset(exported)
assert sum(name.startswith('candidates/') and name.endswith('.joblib') for name in exported)==5
print('Đã tạo:',archive_path)
files.download(archive_path)

Đã tạo: /content/ckd_colab/training_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Kết luận

Có baseline + 4 model; cùng split/pipeline/metric; tuning chỉ trên train; có best params, CV mean/std, train/predict time và size; test còn nguyên cho bước 4.